# Poker

Poker is a much more interesting game to model than Tic Tac Toe as it has hidden states and tactical play which can evolve during a game.

It also has a pretty simple rule set.

I will only consider heads up play for now. If I can solve that perhaps I will look at multiplayer.

The challenge, as ever, will be factorising the problem in a computationally tractable whilst retaining enough relationship information to encode the rules.

> See [this](https://www.google.com/search?sca_esv=2bbc76108a4ecd6f&rlz=1C1YTUH_en-GBGB1056GB1056&sxsrf=ANbL-n70hsan6c1gRscUsyhARbT3Oup-sQ%3A1771241333347&udm=50&aep=26&ntc=1&sa=X&ved=2ahUKEwiDgvGT9N2SAxVZU0EAHSfECVMQ2J8OegQIERAE&biw=1153&bih=801&dpr=1.5&mtid=g_-SaZXRL_mmhbIPl_XIwQk&mstk=AUtExfBxftYtU7jshwECtCnLNxup-10tnIk39Dbbi3vQ3tRxnZVwmBT20gCIjhI_tR83FANYk0JlOfXxXUtvPBWk_fOlH5nsrFs--GiIuOGY2O9ZaEJszJykO7ATAbBcGRiMcnf2_n9QAqQmeDv7AOYl5wPCIiXuiDoZCIqLYAE93o4QlIza3dpiqzEUzaMf-wLOhzTeBJcH4ZIif0JryjYeC6jJlt2J5gAeYSi5LiLLfeC08k0WFumUHgAh7JUlp6Lk6jowZNErzHxMXrZFtpOuoZexJaLHQTpN-pw42gHUmQIVP_v1iV77jCI6Luk5uRe8agwJHBeV8WR3qEDg-Ioak2vjxdKzx1fpzPZPnE49znJ6aVY-IRKOrC9owoVkQXTtto6Q1K_3pwxKagGMk4zu-VnHcNwOtSRRBg&csuir=1&q=poker+hand+scoring+given+board+state&atvm=2) conversation for a chat through the params with Gemini.

## Matrix limitations

- The **A** matrix is Observation -> State, so **knows nothing of the action** that generated its data.

This means if an observation mapping depends on how it was generated, that needs to be somehow encoded in the hidden state.

- The **B** matrix is State -> Action -> State, so it **knows nothing of observations** that can be inferred from the state.

This means if a transition depends on a factor, it *must* be in the hidden state even if it *could* be inferred as an observation from the other states.

- A and B factors can depend on other factors *at the current timestep*, but **not on their future (i.e t+1) state**.

This means 
1. We *can* say 'Given action f our state x will transition to state y depending on the **current state of z**'
2. We *can't* say 'Given action f our state x will transition to state y depending on the **how z transitions**'

e.g. **no way** to express 

'If the game is *transitioning to* the river, my hand strength will change +-'

**only**

'If the game phase *is the* turn then my hand strength will change +-'

- B actions are factor specific - **one action cannot affect multiple factors**.

This means we can't say e.g. ...

1. If you bet, the flop transitions to the turn

*and*

2. If you bet, your relative stack size transitions down

... unless we combine those factors to get a mega factor with game phase * stack size dims which encode the transitions for every given combination of stack size and game phase.

When working through the logic it is easy to be tempted to keep tupling things into a larger and larger single factor as then you can specify every granular transition, but it quickly becomes intractable.

- C matrix preferences can't have dependencies 

This means there is **no way** to say 'I prefer X **if Z, else** I prefer Y'). You can only say what you like on a per factor basis. 

Certain versions of pymdp might allow the C matrix to evolve over time (as preferences change) but this isn't usually the case and still doesn't allow cross factor dependencies.

## Policy guidance

Allowed chains of actions (policies) can be declared when creating the agent, retricting them to selecting valid moves (if sampling_mode=full, see tic-tac-toe notebook).

Filtering policies also drastically reduces the resources required to handle longer policy lengths as we only need to consider legal moves.

## Hidden States

The hidden states are world facts that we try to infer by observing.

Importantly, we **do not** need to encode the actual cards on the board, only their relative values.

### Board state

Boards are commonly referred to as 'Dry' or 'Wet' (or variations thereof). describing how risky they are

- Dry = low risk, not many hands available.
- Wet = high risk, lots of potential winning hands.

This is important because e.g. having a strong hand on a dry board is better (higher *effective* hand strength) than the same hand on a wet board.

This is also directly observable.

### Hand strength(s)

There are many fast hand-scoring libraries that can be used in the environment (generative process) to get this fact.

The floats they generate can be bucketed into strength categories (Weak, Medium, Strong etc etc).

This allows us to manage a hidden state for both

- Our hand strength (directly observable)
- Our opponent's hand strength (indirectly observable - we know how risky the board is and how they've acted, but not what they are holding)

### Opponent play style

This is completely hidden, something the agent will need to infer from its observations.

### Game phase

How your hand strength is going to transition at a given timestep depends on if it is a new phase, as that is when new cards are dealt.

Other timesteps are just betting rounds.

This means we need the state to contain the game phase, which is one of

- Pre-flop (?)
- Flop
- Turn
- River
- Ended (?) (or perhaps 'Win', 'Lose' and 'Split'?)

### Relative stack size

How does your stack size compare to your opponents? Are they roughly the same, or is yours 10X bigger?

The absolute amounts don't matter, just the relative size.

You want to plan actions which get you to states where you have a bigger stack.

## Observations

### Direct senses

As mentioned, we will directly (identity A matrix) observe

- Our hand strength
- Board state
- Relative stack size

### Inferred opponent strength / play style

We will need to infer what opponent action we would likely observe given a hidden hand strength and play style.

- We observe a 'Large Bet' opponent action on a 'Wet' board state

We can infer either

- They have a strong hand (could be either tight or loose)
- They have a weak hand and are loose


## Transitions

### Hand strength

For our hand strength 'null' (uncontrollable) action, we might want to say e.g.

- If my current hand strength is Strong
- And the board is Wet
- And my opponent has a Medium hand
- And the Phase is transitioning (*)
- Then my hand strength might transition to Medium or Low

or

- If my current hand strength is Strong
- And the board is Dry
- And my opponent has a Weak hand
- And the Phase is transitioning this turn (*)
- Then my hand will strength likely remain Strong

For our opponents hand strength transition, we might say

- If their hand strength is Medium
- And the board is Wet
- And I have a Weak hand
- And the phase is transitioning this turn (*)
- Then their hand strength will likely transition to Strong

Here we face an important issue - we **only** expect hand strength to change when the turn phase *changes*, i.e. a new card is dealt.

### Game Phase

As noted in the intro, we can only make transitions depend on how things are *now*.

This leads us to consider pairing up all the possible phases and board states, so we can say e.g.

Flop / Wet -> Turn / Dry = Unlikely

Turn / Medium -> River / Wet = Likely

The trouble with this slippery slope is not only is it inefficient, we soon realise that

- Phase transitions depend not only on how we act, but what the previous opponent action was (Did they check and we call? Next Phase. Did I check as the first actor? Same Phase.)
- Opponent action as discussed above is an observation, not a world state so B doesn't have access to for its logic.
- Relative stack transitions also depend on whether you fold, check, bet or raise, however actions can't be shared across factors.

Before you know it, you have Phase, Board State, Relative Stack and Opponent Action all in one mega-factor and it becomes intractable. 

Putting that to one side...

### Relative stack

- Folding or checking result in the relative stack remaining unchanged
- Call, Bet or Raise result in in it getting smaller.
- The likelihood of those actions leading to you winning and your stack growing is proportional to your hand strength, the opponent's hand strength and the board state.
- As mentioned, we might want to execute the Fold / Check / Call / Bet / Raise actions on this factor, but we already decided we need them for phase transition logic...
